# CRBN Molecular-Glue-Degrader (MGD) Screening Model for PPIL4

Built on top of `02_screening_model_ipynb_(ryan).py`. Pools ChEMBL CRBN single-protein and CRBN/neosubstrate ternary-complex bioactivity data (CyclinK, BCL2, HDAC4), dedupes by compound, and trains a Random Forest to predict CRBN-glue-competent chemotype -- a pre-filter for candidate PPIL4-targeting MGD warheads. See docstrings in each cell for caveats: this predicts CRBN-glue chemistry plausibility, not confirmed PPIL4 degradation (no such data exists publicly yet).

## Step 1: Pooled CRBN Molecular-Glue Data Pipeline (ChEMBL)

In [ ]:
# -*- coding: utf-8 -*-
"""
CRBN Molecular-Glue-Degrader (MGD) Screening Model -- for PPIL4 MGD design
============================================================================
Goal: build a Random Forest classifier that predicts whether a small
molecule has a "CRBN-glue-competent" chemotype -- i.e. the chemistry
needed for the CRBN-recruiting half of a molecular glue degrader (MGD).

Why this, and not a direct "PPIL4 degrader" model
---------------------------------------------------
ChEMBL has essentially no PPIL4 bioactivity data usable for ML: as of this
pull, PPIL4 (CHEMBL5725153) has only 6 activity records, all from a single
compound (Molibresib) in a chemoproteomic dose-response panel. There is no
known CRBN-PPIL4 molecular glue in any public database -- that's exactly
the novel hypothesis this project is testing, so by definition no training
data for "degrades PPIL4" can exist yet.

What *does* exist: ChEMBL curates several CRBN/neosubstrate ternary-complex
targets (CRBN-CyclinK, CRBN-BCL2, CRBN-HDAC4, ...) in addition to the plain
CRBN single-protein target. These come from real molecular glue degrader
programs (e.g. CDK12/CyclinK glues, BCL2 glues) and are labeled with
degradation-relevant readouts (DC50, Emax, EC50), not just simple binding
affinity. Pooling these gives a much better proxy dataset for "CRBN-glue
pharmacophore" than plain CRBN-binder data alone.

This model answers: "does this candidate molecule have the chemistry of a
compound capable of engaging CRBN in a glue-type ternary complex?" It is a
triage filter for the CRBN-recruiting warhead/chemotype of a prospective
PPIL4-targeting MGD -- NOT a prediction that a given molecule will degrade
PPIL4. Actually identifying CRBN-PPIL4 glues requires the structure-based
ternary-complex modeling (CRBN surface + PPIL4 surface complementarity)
described in the project's computational phase; this RF (chemical
fingerprint only, no target structure) is complementary to that, not a
replacement for it. Use it to (a) pre-filter virtual libraries down to
CRBN-glue-plausible chemotypes before expensive docking, and/or (b)
sanity-check structure-based PPIL4 leads for CRBN-compatible chemistry
before committing to wet-lab synthesis.

Outputs:
    X : np.ndarray, shape (n_unique_compounds, 2048)  -- Morgan fingerprints
    Y : np.ndarray, shape (n_unique_compounds,)        -- glue-competent label (0/1)
Saved to disk as X_crbn_glue_fingerprints.npy / Y_crbn_glue_labels.npy
"""

import requests
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

CHEMBL_BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"
FINGERPRINT_BITS = 2048
FINGERPRINT_RADIUS = 2

# Potency/effect thresholds for calling a record "active" (glue/binder-competent)
POTENCY_THRESHOLD_NM = 10000   # DC50 / EC50 / IC50 / Ki / Kd <= 10 uM -> active
EFFECT_THRESHOLD_PCT = 50.0    # Emax / Activity / Inhibition (%) >= 50% -> active

# CRBN and CRBN/neosubstrate ternary-complex targets with usable record counts
# (identified by querying ChEMBL target/search for "CRBN" / "cereblon" and
# checking activity counts per target; targets with <10 records dropped as
# single-compound chemoproteomic panel noise, not real SAR).
TARGET_IDS = {
    "CHEMBL3763008": "CRBN (single protein)",
    "CHEMBL4523685": "CRBN / BCL2 (glue ternary complex)",
    "CHEMBL4296102": "CRBN / Casein kinase I alpha (glue ternary complex)",
    "CHEMBL4296127": "CRBN / HDAC4 (glue ternary complex)",
}


def fetch_all_activities(target_chembl_id, batch_size=1000):
    url = f"{CHEMBL_BASE_URL}/activity"
    all_records = []
    offset = 0
    while True:
        params = {
            "target_chembl_id": target_chembl_id,
            "format": "json",
            "limit": batch_size,
            "offset": offset,
        }
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        activities = data.get("activities", [])
        if not activities:
            break
        all_records.extend(activities)

        total = data["page_meta"]["total_count"]
        offset += batch_size
        if offset >= total:
            break
    return all_records


def label_record(record, potency_threshold_nm=POTENCY_THRESHOLD_NM,
                  effect_threshold_pct=EFFECT_THRESHOLD_PCT):
    """
    Label a single ChEMBL activity record as glue/binder-active (1),
    inactive (0), or unlabelable (None).

    Priority order (most to least informative for glue pharmacology):
      1. Degradation/binding potency in nM (DC50, EC50, IC50, Ki, Kd)
      2. Percent effect (Emax, Activity, Inhibition) at reported dose
      3. Curated activity_comment text ("active"/"inactive")
    """
    std_type = (record.get("standard_type") or "").upper()
    std_value = record.get("standard_value")
    std_units = (record.get("standard_units") or "").lower()

    potency_types = {"DC50", "EC50", "EC90", "IC50", "KI", "KD"}
    if std_type in potency_types and std_value is not None and std_units == "nm":
        try:
            value = float(std_value)
        except (TypeError, ValueError):
            return None
        return 1 if value <= potency_threshold_nm else 0

    percent_types = {"EMAX", "ACTIVITY", "INHIBITION"}
    if std_type in percent_types and std_value is not None and std_units in ("%", ""):
        try:
            value = float(std_value)
        except (TypeError, ValueError):
            return None
        return 1 if value >= effect_threshold_pct else 0

    comment = (record.get("activity_comment") or "").lower()
    if "active" in comment and "inactive" not in comment:
        return 1
    if "inactive" in comment:
        return 0

    return None


def smiles_to_fingerprint(smiles, n_bits=FINGERPRINT_BITS, radius=FINGERPRINT_RADIUS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def build_dataset():
    """
    Pull activities from every target in TARGET_IDS, dedupe by canonical
    SMILES (majority-vote the label across duplicate/repeat records so the
    same compound can't land in both train and test with conflicting or
    redundant rows), and return (compounds, X, Y).
    """
    from collections import defaultdict

    label_votes = defaultdict(list)  # canonical_smiles -> [label, label, ...]

    for target_id, description in TARGET_IDS.items():
        print(f"Fetching {target_id} ({description}) ...")
        records = fetch_all_activities(target_id)
        print(f"  {len(records)} raw activity records")

        kept = 0
        for record in records:
            smiles = record.get("canonical_smiles")
            if not smiles:
                continue
            label = label_record(record)
            if label is None:
                continue
            label_votes[smiles].append(label)
            kept += 1
        print(f"  {kept} labelable records")

    print(f"\nUnique compounds across all targets: {len(label_votes)}")

    compounds, X_rows, Y_labels = [], [], []
    skipped_bad_smiles = 0
    for smiles, votes in label_votes.items():
        fp = smiles_to_fingerprint(smiles)
        if fp is None:
            skipped_bad_smiles += 1
            continue
        # majority vote across all records for this compound; ties -> active
        # (glue chemotypes are rare positives, so ties lean toward keeping them)
        label = 1 if sum(votes) >= len(votes) / 2 else 0
        compounds.append(smiles)
        X_rows.append(fp)
        Y_labels.append(label)

    X = np.array(X_rows)
    Y = np.array(Y_labels)

    print(f"Skipped (invalid SMILES): {skipped_bad_smiles}")
    print(f"\nFinal dataset: X {X.shape}, Y {Y.shape}")
    print(f"Glue/binder-active (Y=1): {int(Y.sum())}")
    print(f"Inactive (Y=0): {int((Y == 0).sum())}")

    return compounds, X, Y


if __name__ == "__main__":
    compounds, X, Y = build_dataset()
    np.save("X_crbn_glue_fingerprints_(Ryan).npy", X)
    np.save("Y_crbn_glue_labels_(Ryan).npy", Y)
    with open("crbn_glue_compounds_(Ryan).txt", "w") as f:
        f.write("\n".join(compounds))
    print("\nSaved X_crbn_glue_fingerprints_(Ryan).npy, Y_crbn_glue_labels_(Ryan).npy, crbn_glue_compounds_(Ryan).txt")

## Step 2: CRBN Glue-Chemotype Random Forest Classifier

In [ ]:
# -*- coding: utf-8 -*-
"""
CRBN Glue-Chemotype Classifier -- Random Forest, 80/20 Split with LOOCV
=========================================================================
Trains on the pooled CRBN / CRBN-neosubstrate-ternary-complex dataset built
by 03_mgd_ppil4_crbn_screening.py (686 unique compounds, deduped by
canonical SMILES so no compound can leak across train/test or across LOOCV
folds).

Approach (same structure as the original CRBN binder script):
  1. 80/20 train/test split (test set untouched until final evaluation).
  2. LOOCV within the 80% training set for a robust internal AUC estimate.
  3. Train a final Random Forest on the full 80% training set.
  4. Evaluate once on the untouched 20% test set.

Then: load a list of candidate SMILES (e.g. the ~20 structure-based PPIL4
leads from the docking phase, or known CRBN-glue chemotypes such as
glutarimide/isoindolinone scaffolds you're considering as the E3-recruiting
arm) and score each for predicted CRBN-glue-chemotype probability.

IMPORTANT CAVEAT: this model predicts "does this molecule look like known
CRBN-glue chemistry", not "will this molecule degrade PPIL4". No CRBN-PPIL4
ternary complex data exists publicly (that's the novel hypothesis this
project tests). Use the probability output as a chemistry-plausibility
filter alongside, not instead of, the structure-based CRBN-PPIL4 ternary
docking work.
"""

import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, LeaveOneOut
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix

RDLogger.DisableLog("rdApp.*")

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_ESTIMATORS = 200
FINGERPRINT_BITS = 2048
FINGERPRINT_RADIUS = 2


def load_data():
    X = np.load("X_crbn_glue_fingerprints_(Ryan).npy")
    Y = np.load("Y_crbn_glue_labels_(Ryan).npy")
    return X, Y


def smiles_to_fingerprint(smiles, n_bits=FINGERPRINT_BITS, radius=FINGERPRINT_RADIUS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def run_loocv(X_train, y_train):
    loo = LeaveOneOut()
    n = X_train.shape[0]

    y_true = np.zeros(n, dtype=int)
    y_pred = np.zeros(n, dtype=int)
    y_proba = np.zeros(n, dtype=float)

    for i, (tr_idx, te_idx) in enumerate(loo.split(X_train)):
        clf = RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        )
        clf.fit(X_train[tr_idx], y_train[tr_idx])

        y_true[te_idx[0]] = y_train[te_idx[0]]
        y_proba[te_idx[0]] = clf.predict_proba(X_train[te_idx])[:, 1][0]
        y_pred[te_idx[0]] = clf.predict(X_train[te_idx])[0]

        if (i + 1) % 100 == 0 or (i + 1) == n:
            print(f"  LOOCV progress: {i + 1}/{n}")

    return y_true, y_pred, y_proba


def evaluate(y_true, y_pred, y_proba, label=""):
    print(f"\nClassification report ({label}):")
    print(classification_report(y_true, y_pred, target_names=["Non-glue", "Glue-competent"]))

    print(f"Confusion matrix ({label}):")
    print(confusion_matrix(y_true, y_pred))

    auc_score = roc_auc_score(y_true, y_proba)
    print(f"\n{label} AUC score: {auc_score:.4f}")

    return auc_score


def plot_roc(y_true, y_proba, auc_score, title, filename):
    fpr, tpr, _ = roc_curve(y_true, y_proba)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {auc_score:.3f})")
    plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Random guess")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    print(f"Saved ROC curve to {filename}")


def score_candidates(clf, smiles_list):
    """
    Score a list of candidate SMILES for predicted CRBN-glue-chemotype
    probability. Returns list of (smiles, probability_or_None).
    None means the SMILES failed to parse.
    """
    results = []
    for smi in smiles_list:
        fp = smiles_to_fingerprint(smi)
        if fp is None:
            results.append((smi, None))
            continue
        proba = clf.predict_proba(fp.reshape(1, -1))[:, 1][0]
        results.append((smi, proba))
    return results


if __name__ == "__main__":
    X, Y = load_data()
    print(f"Loaded X: {X.shape}, Y: {Y.shape}\n")

    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=Y
    )
    print(f"Train set: {X_train.shape[0]} samples ({1 - TEST_SIZE:.0%})")
    print(f"Test set:  {X_test.shape[0]} samples ({TEST_SIZE:.0%})\n")

    print(f"Running LOOCV on {X_train.shape[0]} training samples...")
    loo_true, loo_pred, loo_proba = run_loocv(X_train, y_train)
    loo_auc = evaluate(loo_true, loo_pred, loo_proba, label="Training LOOCV")
    plot_roc(
        loo_true, loo_proba, loo_auc,
        title="ROC Curve -- CRBN Glue Chemotype (Training Set LOOCV)",
        filename="crbn_glue_roc_curve_train_loocv_(Ryan).png",
    )

    final_clf = RandomForestClassifier(
        n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
    )
    final_clf.fit(X_train, y_train)

    test_pred = final_clf.predict(X_test)
    test_proba = final_clf.predict_proba(X_test)[:, 1]
    test_auc = evaluate(y_test, test_pred, test_proba, label="Held-out Test Set")
    plot_roc(
        y_test, test_proba, test_auc,
        title="ROC Curve -- CRBN Glue Chemotype (Held-out 20% Test Set)",
        filename="crbn_glue_roc_curve_test_(Ryan).png",
    )

    print("\n=== Summary ===")
    print(f"Training LOOCV AUC: {loo_auc:.4f}")
    print(f"Held-out Test AUC:  {test_auc:.4f}")

    # --- Example: score candidate PPIL4-MGD warheads for CRBN-glue chemotype ---
    # Replace with your actual structure-based PPIL4 lead SMILES once available.
    # Included here are well-known CRBN-glue degron scaffolds (thalidomide,
    # lenalidomide, pomalidomide) as a sanity check -- they should score high.
    example_candidates = {
        "Thalidomide": "O=C1CCC(N2C(=O)c3ccccc3C2=O)C(=O)N1",
        "Lenalidomide": "NC1=CC=CC2=C1C(=O)N(C1CCC(=O)NC1=O)C2",
        "Pomalidomide": "NC1=CC=CC2=C1C(=O)N(C1CCC(=O)NC1=O)C2=O",
    }
    print("\n=== Example candidate scoring (sanity check: known CRBN glues) ===")
    scored = score_candidates(final_clf, list(example_candidates.values()))
    for (name, _), (smi, proba) in zip(example_candidates.items(), scored):
        p_str = f"{proba:.3f}" if proba is not None else "invalid SMILES"
        print(f"  {name}: P(CRBN-glue-competent) = {p_str}")